# Selección de Región para Nuevos Pozos Petroleros — OilyGiant

Este proyecto tiene como objetivo identificar la región más rentable para abrir **200 nuevos pozos petroleros**, utilizando modelos de regresión lineal y análisis de riesgo mediante la técnica de *bootstrapping*.

## Estructura del proyecto

1. Carga y preparación de datos
2. Entrenamiento y evaluación del modelo por región
3. Preparación para el cálculo de ganancias
4. Cálculo de ganancia por región
5. Análisis de riesgos con bootstrapping

## 1. Carga y Preparación de Datos

Se cargan los tres conjuntos de datos correspondientes a las regiones geológicas disponibles.
Se revisa la estructura general, tipos de datos, valores ausentes y posibles duplicados.

In [28]:
# importaciones
import pandas as pd
import numpy as np


In [29]:
# carga de datos
geo_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_2 = pd.read_csv('/datasets/geo_data_2.csv')

# exploración inicial
print('*** EXPLORACIÓN INICIAL DE LAS TRES REGIONES ***')
print()
for nombre, df in [('Región 0', geo_0), ('Región 1', geo_1), ('Región 2', geo_2)]:
    print(f'=== {nombre} ===')
    print(df.info())
    print(df.head())
    print(f'Duplicados: {df.duplicated().sum()}')
    print(f'Valores nulos:\n{df.isnull().sum()}')
    print()

*** EXPLORACIÓN INICIAL DE LAS TRES REGIONES ***

=== Región 0 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
Duplicados: 0
Valores nulos:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

=== Región 1 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entrie

### 1.1 Discusión — Carga y Preparación de Datos

Los tres conjuntos de datos presentan la misma estructura: **100,000 registros** y **5 columnas** cada uno.

- **id**: identificador único del pozo (tipo objeto, no se usará como variable predictora)
- **f0, f1, f2**: características geológicas numéricas (float64). No se sabe exactaemnte qué miden, pero es importante el poder predictivo sobre el volumen de reservas. 
- **product**: volumen de reservas en miles de barriles (variable objetivo)

**Calidad de los datos:**
- No se encontraron valores nulos en ninguna región
- No se encontraron registros duplicados
- Los tipos de datos son correctos para el modelado

**Observación:** Los rangos de las características varían entre regiones.
La Región 1 presenta valores considerablemente más amplios en `f0` y `f1` (ej. f0 entre -15 y +14), mientras que las Regiones 0 y 2 tienen rangos más acotados.
Esto sugiere distribuciones distintas entre regiones, aunque no afecta la validez del modelo de regresión lineal.
Considerando las características geológicas, el modelo de regresión lineal va a parender qué combinación de estas tres características explica mejor cuánto petróleo hay en un pozo. 

Los datos están listos para la etapa de modelado.

## 2. Entrenamiento y Evaluación del Modelo

Se entrena un modelo de **regresión lineal** para cada región.
Los datos se dividen en conjunto de entrenamiento (75%) y validación (25%).
Se evalúa el desempeño del modelo con la raíz del error cuadrático medio, **RMSE**, y el volumen medio de reservas predicho.

In [30]:
# importaciones
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# función de entrenamiento y evaluación
def entrenar_modelo(df, nombre):
    
    # features y objetivo
    X = df[['f0', 'f1', 'f2']]
    y = df['product']
    
    # división de datos
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.25, random_state=42
    )
    
    # entrenamiento
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    
    # predicciones
    predicciones = pd.Series(modelo.predict(X_valid), index=y_valid.index)
    
    # métricas
    rmse = mean_squared_error(y_valid, predicciones) ** 0.5
    media_pred = predicciones.mean()
    
    print(f'=== {nombre} ===')
    print(f'Volumen medio predicho: {media_pred:.2f} miles de barriles')
    print(f'RMSE: {rmse:.2f}')
    print()
    
    return predicciones, y_valid

# ejecución para las tres regiones
pred_0, valid_0 = entrenar_modelo(geo_0, 'Región 0')
pred_1, valid_1 = entrenar_modelo(geo_1, 'Región 1')
pred_2, valid_2 = entrenar_modelo(geo_2, 'Región 2')

=== Región 0 ===
Volumen medio predicho: 92.40 miles de barriles
RMSE: 37.76

=== Región 1 ===
Volumen medio predicho: 68.71 miles de barriles
RMSE: 0.89

=== Región 2 ===
Volumen medio predicho: 94.77 miles de barriles
RMSE: 40.15



### 2.1 Discusión — Entrenamiento y Evaluación del Modelo

El proceso de entrenamiento y validación se encapsuló en la función `entrenar_modelo()`, aplicada de forma idéntica a las tres regiones, evitando duplicación de código tal como lo requieren las instrucciones del proyecto.

| Región | Volumen medio predicho (miles de barriles) | RMSE  |
|--------|--------------------------------------------|-------|
| 0      | 92.40                                      | 37.76 |
| 1      | 68.71                                      | 00.89 |
| 2      | 94.77                                      | 40.15 |

**Hallazgos:**

- La **Región 1** presenta el RMSE más bajo con diferencia (0.89 miles de barriles), lo que indica que el modelo predice con alta precisión en esa región. Sus predicciones son muy confiables. Esto sugiere que la relación entra las características y el volumen de reservas es casi perfectamente lineal, los rangos amplios (paso 1), no generan ruido. Sin embargo, tiene un volumen promedio predicho más bajo (68.71), lo que puee jugarle en contra en terminos de ganancia bruta.
  
- Las **Regiones 0 y 2** tienen RMSE elevados (~37-40 miles de barriles), lo que sugiere mayor variabilidad en los datos y predicciones menos precisas. Tienen los rangos más acotados pero hay mucho más ruido, lo que indica que las características no explican tan limpiamente el volumen de reservas. Es probable que estas regiones muestren mayor variabilidad en sus distribuciones de ganancia (por confirmar más adelante).
- En cuanto al volumen medio predicho, las Regiones 0 y 2 superan a la Región 1, pero esa ventaja debe evaluarse con cautela dado el alto error de sus modelos.

Estos resultados son un primer indicador, la decisión final sobre qué región desarrollar se tomará tras el análisis de ganancias y riesgos. La pregunta que tenemos es: ¿conviene más una región predecible con menor volumen, o una región con mayor volumen pero más incierta?. Esto se evaluará en el análisis de riesgo con bootstrapping. 

## 3. Preparación para el Cálculo de Ganancias

Se definen las constantes del negocio y se establece el volumen mínimo de reservas que debe producir cada pozo para que la inversión sea rentable.
Se compara este umbral con el volumen medio predicho por región.

In [31]:
# constantes del negocio
PRESUPUESTO = 100_000_000       # presupuesto total en USD
POZOS = 200                     # número de pozos a desarrollar
INGRESO_POR_UNIDAD = 4_500      # USD por cada mil barriles
PUNTOS_EXPLORADOS = 500         # puntos estudiados por región

# volumen mínimo por pozo para no tener pérdidas
costo_por_pozo = PRESUPUESTO / POZOS
volumen_minimo = costo_por_pozo / INGRESO_POR_UNIDAD

print(f'Presupuesto total:         ${PRESUPUESTO:,.0f}')
print(f'Costo por pozo:            ${costo_por_pozo:,.0f}')
print(f'Volumen mínimo por pozo:   {volumen_minimo:.2f} miles de barriles')
print()

# comparación con volumen medio predicho por región
print('=== Comparación con volumen medio predicho ===')
for nombre, pred in [('Región 0', pred_0), ('Región 1', pred_1), ('Región 2', pred_2)]:
    media = pred.mean()
    diferencia = media - volumen_minimo
    estado = 'supera umbral' if diferencia > 0 else 'no supera umbral'
    print(f'{nombre}: {media:.2f} miles de barriles | {estado} (diferencia: {diferencia:.2f})')

Presupuesto total:         $100,000,000
Costo por pozo:            $500,000
Volumen mínimo por pozo:   111.11 miles de barriles

=== Comparación con volumen medio predicho ===
Región 0: 92.40 miles de barriles | no supera umbral (diferencia: -18.71)
Región 1: 68.71 miles de barriles | no supera umbral (diferencia: -42.40)
Región 2: 94.77 miles de barriles | no supera umbral (diferencia: -16.34)


### 3.1 Discusión — Preparación para el Cálculo de Ganancias

Para que la inversión sea rentable, cada pozo debe producir un mínimo de **111.11 miles de barriles**. Al comparar este umbral con el volumen medio predicho por región, ninguna región lo supera:

| Región | Volumen medio predicho | Diferencia vs umbral |
|--------|------------------------|----------------------|
| 0      | 92.40 miles de barriles  | -18.71              |
| 1      | 68.71 miles de barriles  | -42.40              |
| 2      | 94.77 miles de barriles  | -16.34              |

Esto **no significa que las regiones sean inviables**. El volumen medio considera todos los pozos explorados, incluyendo los menos productivos.

**Para el cálculo de ganancias se seguirá esta estrategia:**

1. De los 500 puntos explorados por región, se seleccionarán los **200 pozos con mayor volumen predicho** por el modelo.
2. La ganancia se calculará usando el **volumen real** de esos 200 pozos, no sus predicciones.
3. Al enfocarse en los pozos más prometedores, se espera superar el umbral de rentabilidad de 111.11 miles de barriles por pozo.

Este enfoque simula la decisión real de negocio: explorar amplio (500 puntos) y desarrollar solo los mejores (200 pozos).

## 4. Cálculo de Ganancia por Región

Se define una función para calcular la ganancia potencial de cada región.
El proceso sigue esta lógica:

1. El modelo ya realizó predicciones sobre el conjunto de validación (25,000 pozos por región).
2. Se identifican los **200 pozos con mayor volumen predicho** usando esas predicciones.
3. Se almacenan las predicciones de esos 200 pozos en variables separadas (`top_0`, `top_1`, `top_2`) como referencia del paso de selección.
4. La ganancia se calcula con el **volumen real** de esos 200 pozos, no con el predicho — simulando lo que ocurriría si se perforaran exactamente esos pozos.

**Nota:** Las variables `top_0`, `top_1`, `top_2` documentan qué pozos seleccionaría el modelo en condiciones ideales. Sin embargo, el análisis de riesgo en el paso 5 requiere simular escenarios reales de exploración,
por lo que el *bootstrapping* muestrea del conjunto de validación completo (25,000 pozos), no de estos 200.

In [32]:
# función de cálculo de ganancia
def calcular_ganancia(predicciones, valores_reales):
    
    # selección de los 200 pozos con mayor predicción
    indices_top = predicciones.nlargest(POZOS).index
    
    # volumen real de esos 200 pozos
    volumen_real = valores_reales[indices_top].sum()
    
    # ganancia
    ganancia = volumen_real * INGRESO_POR_UNIDAD - PRESUPUESTO
    
    return ganancia

# almacenamiento de predicciones top 200 por región
top_0 = pred_0.nlargest(POZOS)
top_1 = pred_1.nlargest(POZOS)
top_2 = pred_2.nlargest(POZOS)

# cálculo por región
print('=== Ganancia potencial por región (200 mejores pozos) ===')
for nombre, top, valid in [
    ('Región 0', top_0, valid_0),
    ('Región 1', top_1, valid_1),
    ('Región 2', top_2, valid_2)
]:
    ganancia = calcular_ganancia(top, valid)
    print(f'{nombre}: ${ganancia:,.2f}')

=== Ganancia potencial por región (200 mejores pozos) ===
Región 0: $33,591,411.14
Región 1: $24,150,866.97
Región 2: $25,985,717.59


Se identifican la distribución de los pozos top por cada región.

In [33]:
# distribución de los 200 pozos top por región
print('=== Estadísticas de los 200 pozos top por región ===\n')
for nombre, pred, valid in [
    ('Región 0', top_0, valid_0),
    ('Región 1', top_1, valid_1),
    ('Región 2', top_2, valid_2)
]:
    indices_top = pred.nlargest(POZOS).index
    volumen_top = valid[indices_top]
    
    print(f'{nombre}:')
    print(f'  Volumen real mínimo:   {volumen_top.min():.2f} miles de barriles')
    print(f'  Volumen real máximo:   {volumen_top.max():.2f} miles de barriles')
    print(f'  Volumen real promedio: {volumen_top.mean():.2f} miles de barriles')
    print(f'  Pozos sobre umbral (111.11): {(volumen_top >= volumen_minimo).sum()} de {POZOS}')
    print()

=== Estadísticas de los 200 pozos top por región ===

Región 0:
  Volumen real mínimo:   65.39 miles de barriles
  Volumen real máximo:   184.61 miles de barriles
  Volumen real promedio: 148.43 miles de barriles
  Pozos sobre umbral (111.11): 186 de 200

Región 1:
  Volumen real mínimo:   137.95 miles de barriles
  Volumen real máximo:   137.95 miles de barriles
  Volumen real promedio: 137.95 miles de barriles
  Pozos sobre umbral (111.11): 200 de 200

Región 2:
  Volumen real mínimo:   47.58 miles de barriles
  Volumen real máximo:   189.12 miles de barriles
  Volumen real promedio: 139.98 miles de barriles
  Pozos sobre umbral (111.11): 165 de 200



### 4.1 Discusión — Cálculo de Ganancia por Región

Al seleccionar los 200 pozos con mayor volumen predicho, los resultados de ganancia potencial y distribución de volumen real son los siguientes:

| Región | Ganancia potencial | Promedio real | Pozos sobre umbral |
|--------|--------------------|---------------|--------------------|
| 0      | $33,591,411.14     | 148.43 mb     | 186 de 200         |
| 1      | $24,150,866.97     | 137.95 mb     | 200 de 200         |
| 2      | $25,985,717.59     | 139.98 mb     | 165 de 200         |

**Hallazgos por región:**

- **Región 0:** Mayor ganancia potencial ($33.6M). El modelo selecciona bien, aunque con cierto ruido: 14 pozos no superan el umbral y el volumen mínimo real es de 65.39 mb. Su RMSE alto se refleja en esta variabilidad.

- **Región 1:** Todos los pozos superan el umbral y presentan exactamente el mismo volumen real (137.95 mb), lo que confirma la relación casi perfectamente lineal detectada en el RMSE de 0.89. Es la región más predecible, aunque no la de mayor ganancia.

- **Región 2:** Promedio real competitivo (139.98 mb), pero solo 165 de 200 pozos superan el umbral y el volumen mínimo real es de 47.58 mb, el más bajo de las tres regiones. Su alta variabilidad representa un riesgo.

**Conclusión preliminar:** Con base únicamente en ganancia potencial, la **Región 0** sería la elección. Sin embargo, esta decisión no considera el riesgo asociado a cada región. El análisis de *bootstrapping* en el
paso 5 permitirá validar o replantear esta elección con mayor solidez.

## 5. Análisis de Riesgos con Bootstrapping

Se aplica la técnica de *bootstrapping* para estimar la distribución de ganancias de cada región y evaluar el riesgo de pérdidas.

El proceso simula **1,000 veces** el escenario real de exploración:
1. Se toma una muestra aleatoria de **500 puntos** del conjunto de validación completo.
2. Se seleccionan los **200 con mayor predicción**.
3. Se calcula la ganancia con el volumen real de esos 200 pozos.
4. Se repite 1,000 veces para obtener una distribución de ganancias.

**Nota:** Aunque en el paso 4 se almacenaron las predicciones de los 200 mejores pozos (`top_0`, `top_1`, `top_2`), el *bootstrapping* utiliza el conjunto de validación completo (25,000 pozos por región). Esto es necesario para simular correctamente la exploración de 500 puntos aleatorios, que es el escenario real del negocio. Usar solo los 200 top inflaría artificialmente las ganancias.

Con esta distribución se calcula:
- **Ganancia promedio**
- **Intervalo de confianza del 95%**
- **Riesgo de pérdidas** (probabilidad de ganancia negativa)

Solo se considerarán viables las regiones con riesgo de pérdidas **menor al 2.5%**.

In [34]:
# función de bootstrapping
def bootstrapping(predicciones, valores_reales, n_muestras=1000):
    
    state = np.random.RandomState(42)
    ganancias = []
    
    for _ in range(n_muestras):
        
        # muestra aleatoria de 500 puntos
        muestra_pred = predicciones.sample(n=PUNTOS_EXPLORADOS, replace=True, random_state=state)
        muestra_real = valores_reales[muestra_pred.index]
        
        # ganancia de los 200 mejores
        ganancia = calcular_ganancia(muestra_pred, muestra_real)
        ganancias.append(ganancia)
    
    ganancias = pd.Series(ganancias)
    return ganancias

# resultados por región
print('=== Análisis de riesgo por región ===\n')
for nombre, pred, valid in [
    ('Región 0', pred_0, valid_0),
    ('Región 1', pred_1, valid_1),
    ('Región 2', pred_2, valid_2)
]:
    ganancias = bootstrapping(pred, valid)
    
    promedio = ganancias.mean()
    ic_inferior = ganancias.quantile(0.025)
    ic_superior = ganancias.quantile(0.975)
    riesgo = (ganancias < 0).mean() * 100
    
    viabilidad = 'VIABLE' if riesgo <= 2.5 else 'NO VIABLE'
    
    print(f'{nombre}: {viabilidad}')
    print(f'  Ganancia promedio:          ${promedio:,.2f}')
    print(f'  Intervalo de confianza 95%: ${ic_inferior:,.2f} — ${ic_superior:,.2f}')
    print(f'  Riesgo de pérdidas:         {riesgo:.2f}%')
    print()

=== Análisis de riesgo por región ===

Región 0: VIABLE
  Ganancia promedio:          $6,061,226.32
  Intervalo de confianza 95%: $100,894.12 — $12,463,709.81
  Riesgo de pérdidas:         2.50%

Región 1: VIABLE
  Ganancia promedio:          $6,651,176.54
  Intervalo de confianza 95%: $1,808,515.85 — $12,057,104.61
  Riesgo de pérdidas:         0.20%

Región 2: NO VIABLE
  Ganancia promedio:          $5,851,036.38
  Intervalo de confianza 95%: $-8,369.42 — $12,120,508.98
  Riesgo de pérdidas:         2.60%



### 5.1 Discusión — Análisis de Riesgos con Bootstrapping

Tras simular 1,000 escenarios de exploración por región, los resultados son:

| Región | Ganancia promedio | IC 95% inferior | IC 95% superior | Riesgo | Viabilidad |
|--------|-------------------|-----------------|-----------------|--------|------------|
| 0      | $6,061,226.32     | $100,894.12     | $12,463,709.81  | 2.50%  | ✓ VIABLE   |
| 1      | $6,651,176.54     | $1,808,515.85   | $12,057,104.61  | 0.20%  | ✓ VIABLE   |
| 2      | $5,851,036.38     | -$8,369.42      | $12,120,508.98  | 2.60%  | ✗ NO VIABLE|

**Análisis por región:**

- **Región 0:** Viable con riesgo exactamente en el límite permitido (2.50%). Su intervalo de confianza inferior es positivo pero muy cercano a cero,lo que indica poco margen de seguridad.

- **Región 1:** La región más sólida. Riesgo de pérdidas mínimo (0.20%), ganancia promedio más alta ($6,651,176) e intervalo de confianza completamente positivo. Incluso en los peores escenarios genera ganancia.

- **Región 2:** No viable. Su riesgo de pérdidas (2.60%) supera el umbral permitido del 2.5% y su intervalo de confianza inferior es negativo, confirmando pérdidas en algunos escenarios.

En el paso 4 la **Región 0** tenía la mayor ganancia potencial ($33.6M). Pero el bootstrapping revela que esa ganancia era optimista: al simular escenarios reales de 500 puntos, la **Región 1** resulta más rentable y segura.

# Conclusión final

**Se recomienda desarrollar la Región 1**. Es la única región que combina la ganancia promedio más alta con el menor riesgo de pérdidas.
Aunque en el paso 4 la **Región 0** mostraba mayor ganancia potencial, el análisis de *bootstrapping* revela que esa ventaja no se sostiene al simular condiciones reales de exploración.

La predictibilidad de la **Región 1**, evidenciada desde el **RMSE de 0.89**, se traduce en una ventaja competitiva real: el modelo identifica con alta precisión los mejores pozos, minimizando el riesgo del negocio.